# Train a License Plate Detector (YOLOv8n → TFLite) — merged dataset v2

Merges four sources into one training set. **See the Citations cell at the bottom — copy it into your GitHub README.**

| # | Dataset | Images | License | File |
|---|---|---|---|---|
| 1 | Car License Plate (Dataset Ninja / MakeML) | 433 | CC0 1.0 | `car-license-plate-DatasetNinja.tar` |
| 2 | Licence plate (Roboflow) | 1,030 | Public Domain | `Licence plate.yolov8.zip` |
| 3 | License Plate Detection Dataset (Kaggle, fareselmenshawii) | ~5,300 | CC0: Public Domain | `archive.zip` |
| 4 | License Plate Recognition Dataset (Kaggle, adilshamim8) | 10,000+ | CC BY 4.0 | `archive(1).zip` |

Run cells top to bottom. Inspection cells print folder structure before each merge — verify before proceeding, since datasets 3 and 4's exact internal layout hasn't been independently confirmed by me.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 6.7 MB/s eta 0:00:00


## 2. Extract all four archives

Assumes all four files are uploaded to `/content/`. If `unzip` prompts about an existing file (e.g. a leftover `data.yaml` from a prior run), type `A` and press Enter to overwrite all.

In [ ]:
import os

for d in ['dataset1_ninja', 'dataset2_roboflow', 'dataset3_kaggle_cc0', 'dataset4_kaggle_attr']:
    os.makedirs(f'/content/{d}', exist_ok=True)

!tar -xf '/content/car-license-plate-DatasetNinja.tar' -C /content/dataset1_ninja/
!unzip -q '/content/Licence plate.yolov8.zip' -d /content/dataset2_roboflow/
!unzip -q '/content/archive.zip' -d /content/dataset3_kaggle_cc0/
!unzip -q '/content/archive(1).zip' -d /content/dataset4_kaggle_attr/

print('Extraction complete')

replace /content/dataset2_roboflow/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Extraction complete


## 3. Inspect all four folder structures

In [ ]:
def print_tree(root, max_depth=3, max_files=3):
    root = str(root)
    for r, dirs, files in os.walk(root):
        level = r.replace(root, '').count(os.sep)
        if level > max_depth:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(r)}/')
        for f in files[:max_files]:
            print(f'{indent}  {f}')
        if len(files) > max_files:
            print(f'{indent}  ... ({len(files)} files total)')

for name, path in [
    ('dataset1_ninja (Supervisely)', '/content/dataset1_ninja'),
    ('dataset2_roboflow (YOLOv8)', '/content/dataset2_roboflow'),
    ('dataset3_kaggle_cc0', '/content/dataset3_kaggle_cc0'),
    ('dataset4_kaggle_attr', '/content/dataset4_kaggle_attr'),
]:
    print(f'=== {name} ===')
    print_tree(path)
    print()

=== dataset1_ninja (Supervisely) ===
dataset1_ninja/
  meta.json
  LICENSE.md
  README.md
  ds/
    ann/
      Cars414.png.json
      Cars159.png.json
      Cars64.png.json
      ... (433 files total)
    img/
      Cars208.png
      Cars353.png
      Cars423.png
      ... (433 files total)

=== dataset2_roboflow (YOLOv8) ===
dataset2_roboflow/
  README.dataset.txt
  data.yaml
  README.roboflow.txt
  valid/
    labels/
      xemay503_jpg.rf.odp4nGYoTU1BNHzINvbK.txt
      CarLongPlateGen3296_jpg.rf.UHtfa4KAIAmaPu02mvHR.txt
      xemayBigPlate181_jpg.rf.2xTjFPZ5Ckzm4osjV0Za.txt
      ... (204 files total)
    images/
      1649292507302_jpg.rf.QQzI3Vc0R02BmKiltZEW.jpg
      Cars211_png_jpg.rf.ypKlw4XOEhFKMO9L0GL8.jpg
      CarLongPlateGen229_jpg.rf.idhjewqH0HVleBukbg4w.jpg
      ... (204 files total)
  train/
    labels/
      CarLongPlateGen3579_jpg.rf.PdS5uTUBqxrvE28ByPeB.txt
      CarLongPlate112_jpg.rf.6RtVtYThbE24Xk8Bpnys.txt
      CarLongPlateGen751_jpg.rf.GgsZ4QU2ZmC5fa1wR38f.txt


## 4. Set up the merged output folder

In [ ]:
from pathlib import Path

OUT_ROOT = '/content/yolo_dataset'
for split in ['train', 'val']:
    Path(f'{OUT_ROOT}/images/{split}').mkdir(parents=True, exist_ok=True)
    Path(f'{OUT_ROOT}/labels/{split}').mkdir(parents=True, exist_ok=True)

print(f'Created {OUT_ROOT}/images and {OUT_ROOT}/labels with train/val subfolders')

Created /content/yolo_dataset/images and /content/yolo_dataset/labels with train/val subfolders


## 5. Merge dataset 1 (Supervisely → YOLO conversion)

85/15 train/val split. Adjust `SRC_ROOT` if step 3 showed a different structure.

In [ ]:
import json, glob, shutil, random

SRC_ROOT = '/content/dataset1_ninja'

ann_files = glob.glob(f'{SRC_ROOT}/**/ann/*.json', recursive=True)
print(f'Found {len(ann_files)} annotation files in dataset 1')
assert len(ann_files) > 0, 'No annotation files found — check SRC_ROOT against step 3 output'

random.seed(42)
random.shuffle(ann_files)
split_idx = int(len(ann_files) * 0.85)
train_files, val_files = ann_files[:split_idx], ann_files[split_idx:]

def convert_supervisely(ann_path, split, prefix='ds1'):
    with open(ann_path) as f:
        data = json.load(f)

    img_height = data['size']['height']
    img_width = data['size']['width']
    img_name = Path(ann_path).stem
    img_src = Path(ann_path).parent.parent / 'img' / img_name
    if not img_src.exists():
        return False

    lines = []
    for obj in data.get('objects', []):
        if obj.get('geometryType') != 'rectangle':
            continue
        (x1, y1), (x2, y2) = obj['points']['exterior']
        x_min, x_max = sorted([x1, x2])
        y_min, y_max = sorted([y1, y2])
        x_center = ((x_min + x_max) / 2) / img_width
        y_center = ((y_min + y_max) / 2) / img_height
        w = (x_max - x_min) / img_width
        h = (y_max - y_min) / img_height
        lines.append(f'0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}')

    if not lines:
        return False

    dst_name = f'{prefix}_{img_name}'
    shutil.copy(img_src, f'{OUT_ROOT}/images/{split}/{dst_name}')
    label_name = Path(dst_name).stem + '.txt'
    with open(f'{OUT_ROOT}/labels/{split}/{label_name}', 'w') as f:
        f.write('\n'.join(lines))
    return True

converted = sum(convert_supervisely(f, 'train') for f in train_files)
converted += sum(convert_supervisely(f, 'val') for f in val_files)
print(f'Dataset 1: merged {converted} images')

Found 433 annotation files in dataset 1
Dataset 1: merged 433 images


## 6. Merge datasets 2–4 (all already in YOLO format)

In [ ]:
def merge_yolo_folder(src_root, split_map, prefix, out_root=OUT_ROOT):
    added = 0
    for src_split, dst_split in split_map.items():
        img_dir = Path(src_root) / src_split / 'images'
        lbl_dir = Path(src_root) / src_split / 'labels'
        if not img_dir.exists():
            continue
        for img_path in img_dir.glob('*'):
            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if not lbl_path.exists():
                continue
            shutil.copy(img_path, f'{out_root}/images/{dst_split}/{prefix}_{img_path.name}')
            shutil.copy(lbl_path, f'{out_root}/labels/{dst_split}/{prefix}_{img_path.stem}.txt')
            added += 1
    return added

SPLIT_MAP = {'train': 'train', 'valid': 'val', 'val': 'val', 'test': 'val'}

ds2_added = merge_yolo_folder('/content/dataset2_roboflow', SPLIT_MAP, prefix='ds2')
print(f'Dataset 2: merged {ds2_added} images')
assert ds2_added > 0, 'Dataset 2: 0 images merged — check structure from step 3'

ds3_added = merge_yolo_folder('/content/dataset3_kaggle_cc0', SPLIT_MAP, prefix='ds3')
print(f'Dataset 3: merged {ds3_added} images')
if ds3_added == 0:
    print('WARNING: 0 images merged from dataset 3 — check step 3 output, the images/labels')
    print('folders may be nested one level deeper than assumed. Adjust the src_root path below and re-run.')

ds4_added = merge_yolo_folder('/content/dataset4_kaggle_attr', SPLIT_MAP, prefix='ds4')
print(f'Dataset 4: merged {ds4_added} images')
if ds4_added == 0:
    print('WARNING: 0 images merged from dataset 4 — check step 3 output and adjust src_root/split_map below and re-run.')

Dataset 2: merged 1030 images
Dataset 3: merged 0 images
folders may be nested one level deeper than assumed. Adjust the src_root path below and re-run.
Dataset 4: merged 0 images


### If dataset 3 or 4 merged 0 images

Re-check the printed structure from step 3, find the actual path to each dataset's `images`/`labels` folders, and re-run with the corrected root, e.g.:
```python
ds3_added = merge_yolo_folder('/content/dataset3_kaggle_cc0/<correct-subfolder>', SPLIT_MAP, prefix='ds3')
```

## 7. Sanity-check combined totals

In [ ]:
for split in ['train', 'val']:
    n_img = len(os.listdir(f'{OUT_ROOT}/images/{split}'))
    n_lbl = len(os.listdir(f'{OUT_ROOT}/labels/{split}'))
    print(f'{split}: {n_img} images, {n_lbl} labels')

train: 1090 images, 1090 labels
val: 373 images, 373 labels


## 8. Write data.yaml

In [ ]:
yaml_content = """path: /content/yolo_dataset
train: images/train
val: images/val

nc: 1
names: ['license_plate']
"""

with open('/content/yolo_dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

path: /content/yolo_dataset
train: images/train
val: images/val

nc: 1
names: ['license_plate']



## 9. Train

**Starting checkpoint:** this Colab session doesn't have your previous `best.pt` from the last training run — if you want to continue from it rather than start fresh, upload it to `/content/` now and change `'yolov8n.pt'` below to `'/content/best.pt'`. Otherwise this trains fresh on the full combined dataset, which is also a reasonable choice given how much the dataset composition has changed.

With 10,000+ combined images, expect this to take considerably longer than prior runs — possibly 2+ hours on the free T4. Confirm Runtime → Change runtime type is set to GPU first, and keep the tab active periodically to avoid a mid-training disconnect.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # change to '/content/best.pt' to continue from your previous checkpoint

model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='plate_detector',
    name='run1'
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c3e4c295f70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

## 10. Export to TFLite

In [ ]:
model.export(format='tflite', imgsz=640)

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert/
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/plate_detector/run1/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 85 packages in 611ms
Prepared 15 packages in 2.79s
Uninstalled 3 packages in 31ms
Installed 15 packages in 93ms
 + ai-edge-litert==2.1.6
 + ai-edge-qu

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1558: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.



LiteRT: starting export with litert_torch 0.9.3...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:12) [ DONE] LiteRT-Torch Convert (+00:12)

(00:00) [START] Write Model to /content/runs/detect/plate_detector/run1/weights/best.tflite

(00:00) [ DONE] Write Model to /content/runs/detect/plate_detector/run1/weights/best.tflite (+00:00)

LiteRT: export success ✅ 19.7s, saved as '/content/runs/detect/plate_detector/run1/weights/best.tflite' (11.7 MB)

Export complete (20.1s)
Results saved to /content/runs/detect/plate_detector/run1/weights/best.tflite
Predict:         yolo predict task=detect model=/content/runs/detect/plate_detector/run1/weights/best.tflite imgsz=640 
Validate:        yolo val task=detect model=/content/runs/detect/plate_detector/run1/weights/best.tflite imgsz=640 data=/content/yolo_dataset/data.yaml  
Visualize:       https://netron.app


PosixPath('/content/runs/detect/plate_detector/run1/weights/best.tflite')

## 11. Rename and download

Check the printed export path from step 10 first — it should match the pattern below, but confirm before running.

In [ ]:
import shutil
from google.colab import files

shutil.copy('/content/runs/detect/plate_detector/run1/weights/best.tflite', '/content/plate_detector.tflite')
files.download('/content/plate_detector.tflite')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Citations

Copy this section into your GitHub README.

---

### Training data attribution

The license plate detection model in this project was trained on a combination of the following datasets:

1. **Car License Plate** — MakeML / Dataset Ninja. CC0 1.0 (Public Domain). https://datasetninja.com/car-license-plate
2. **Licence plate** — Roboflow Universe. Public Domain. https://universe.roboflow.com/licence-plate-detection-wampn/licence-plate-shtqm
3. **License Plate Detection Dataset** — Fares Elmenshawii, Kaggle. CC0: Public Domain. https://www.kaggle.com/datasets/fareselmenshawii/license-plate-dataset
4. **License Plate Recognition Dataset** — Adil Shamim, Kaggle (originally curated via Roboflow Universe). CC BY 4.0. https://www.kaggle.com/datasets/adilshamim8/license-plate-recognition

Datasets 1–3 do not require attribution; #4 is licensed CC BY 4.0 and is credited here per its terms.